# 11 - Model Comparison: PHEME Dataset

Deep-dive comparison between **RoBERTa Fine-Tuned** and **Ollama Zero-Shot (llama3.1)** on the PHEME dataset.

- **Task**: Classify tweets as `not_rumour` vs `rumour`
- **Topics**: Charlie Hebdo attack (2015), Ferguson unrest (2014)
- **Positive class**: `rumour`

## 1. Imports & Config

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
from sklearn.metrics import (
    confusion_matrix, classification_report,
    f1_score, precision_score, recall_score,
    accuracy_score, roc_curve, auc,
    precision_recall_curve
)

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size']  = 11

DATASET      = 'pheme'
LABEL_NAMES  = ['not_rumour', 'rumour']
LABEL_MAP    = {'not_rumour': 0, 'rumour': 1}
POS_LABEL    = 'rumour'
POS_INT      = 1

RESULTS_DIR  = Path('../../results')
PREDS_DIR    = RESULTS_DIR / 'predictions'
FIGS_DIR     = RESULTS_DIR / 'figures'
FIGS_DIR.mkdir(parents=True, exist_ok=True)

COLORS = {'roberta': '#2980b9', 'ollama': '#e67e22'}

print(f"Dataset: {DATASET.upper()}")
print(f"Labels:  {LABEL_NAMES}")
print(f"Positive class: {POS_LABEL}")

## 2. Load Predictions

In [ ]:
roberta_path  = PREDS_DIR / f'{DATASET}_roberta_test_predictions.csv'
zeroshot_path = PREDS_DIR / f'{DATASET}_zeroshot_test_predictions.csv'

df_r = pd.read_csv(roberta_path)
df_z = pd.read_csv(zeroshot_path)

print(f"RoBERTa predictions:  {len(df_r):,} rows")
print(f"Ollama predictions:   {len(df_z):,} rows")

# Extract arrays
y_true_r = df_r['true_label_int'].values
y_pred_r = df_r['pred_label_int'].values
y_prob_r = df_r[f'prob_{POS_LABEL}'].values

y_true_z = df_z['true_label_int'].values
y_pred_z = df_z['pred_label_int'].values
y_conf_z = df_z['confidence'].values

print("\nRoBERTa — label distribution:")
print(pd.Series(y_pred_r).map({0:'not_rumour',1:'rumour'}).value_counts())
print("\nOllama  — label distribution:")
print(pd.Series(y_pred_z).map({0:'not_rumour',1:'rumour'}).value_counts())

## 3. Metrics Summary

In [ ]:
def get_metrics(y_true, y_pred, name):
    return {
        'Model':            name,
        'Accuracy':         accuracy_score(y_true, y_pred),
        'F1 Macro':         f1_score(y_true, y_pred, average='macro'),
        'F1 Weighted':      f1_score(y_true, y_pred, average='weighted'),
        'Precision (Macro)':precision_score(y_true, y_pred, average='macro', zero_division=0),
        'Recall (Macro)':   recall_score(y_true, y_pred, average='macro', zero_division=0),
        f'Precision ({POS_LABEL})': precision_score(y_true, y_pred, pos_label=POS_INT, zero_division=0),
        f'Recall ({POS_LABEL})':    recall_score(y_true, y_pred, pos_label=POS_INT, zero_division=0),
        f'F1 ({POS_LABEL})':        f1_score(y_true, y_pred, pos_label=POS_INT, zero_division=0),
    }

metrics_r = get_metrics(y_true_r, y_pred_r, 'RoBERTa Fine-Tuned')
metrics_z = get_metrics(y_true_z, y_pred_z, 'Ollama Zero-Shot')

df_metrics = pd.DataFrame([metrics_r, metrics_z]).set_index('Model')

print(f"\n{'='*65}")
print(f" PHEME — Full Metrics Comparison")
print(f"{'='*65}")
print(df_metrics.round(4).to_string())

print("\n--- Classification Report: RoBERTa ---")
print(classification_report(y_true_r, y_pred_r, target_names=LABEL_NAMES, digits=4))

print("--- Classification Report: Ollama ---")
print(classification_report(y_true_z, y_pred_z, target_names=LABEL_NAMES, digits=4))

## 4. Confusion Matrices Side-by-Side

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, y_true, y_pred, title, cmap in [
    (axes[0], y_true_r, y_pred_r, 'RoBERTa Fine-Tuned', 'Blues'),
    (axes[1], y_true_z, y_pred_z, 'Ollama Zero-Shot',   'Oranges'),
]:
    cm      = confusion_matrix(y_true, y_pred)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    annot   = np.array([[f"{cm[i,j]}\n({cm_norm[i,j]:.1%})" for j in range(2)] for i in range(2)])
    sns.heatmap(cm_norm, annot=annot, fmt='', cmap=cmap,
                xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES,
                ax=ax, vmin=0, vmax=1, linewidths=0.5)
    ax.set_title(f'PHEME | {title}', fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')

plt.suptitle('PHEME — Confusion Matrices', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(FIGS_DIR / 'pheme_comparison_cm.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Metrics Bar Chart

In [ ]:
plot_metrics = ['F1 Macro', 'Accuracy', f'F1 ({POS_LABEL})', f'Precision ({POS_LABEL})', f'Recall ({POS_LABEL})']

r_vals = [metrics_r[m] for m in plot_metrics]
z_vals = [metrics_z[m] for m in plot_metrics]

x = np.arange(len(plot_metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 5))
bars1 = ax.bar(x - width/2, r_vals, width, label='RoBERTa Fine-Tuned',
               color=COLORS['roberta'], alpha=0.85, edgecolor='black', linewidth=0.5)
bars2 = ax.bar(x + width/2, z_vals, width, label='Ollama Zero-Shot',
               color=COLORS['ollama'], alpha=0.85, edgecolor='black', linewidth=0.5)

for bar in list(bars1) + list(bars2):
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.005, f'{h:.3f}',
            ha='center', va='bottom', fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(plot_metrics, fontsize=10)
ax.set_ylim([0, 1.1])
ax.set_ylabel('Score')
ax.set_title('PHEME — RoBERTa vs Ollama Zero-Shot', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(FIGS_DIR / 'pheme_comparison_bars.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. ROC & Precision-Recall Curves (RoBERTa)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ROC
fpr, tpr, _ = roc_curve(y_true_r, y_prob_r)
roc_auc     = auc(fpr, tpr)
axes[0].plot(fpr, tpr, color=COLORS['roberta'], lw=2, label=f'RoBERTa (AUC = {roc_auc:.3f})')
axes[0].plot([0,1],[0,1], 'k--', lw=1)
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('PHEME — ROC Curve (RoBERTa)', fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

# PR
prec, rec, _ = precision_recall_curve(y_true_r, y_prob_r)
pr_auc       = auc(rec, prec)
axes[1].plot(rec, prec, color=COLORS['roberta'], lw=2, label=f'RoBERTa (AUC = {pr_auc:.3f})')
axes[1].axhline(y=y_true_r.mean(), color='gray', linestyle='--', lw=1, label='Baseline')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('PHEME — Precision-Recall Curve (RoBERTa)', fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(FIGS_DIR / 'pheme_roberta_roc_pr.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"ROC AUC: {roc_auc:.4f} | PR AUC: {pr_auc:.4f}")

## 7. Error Analysis — Disagreements Between Models

In [ ]:
text_col = 'cleaned_tweet' if 'cleaned_tweet' in df_r.columns else df_r.columns[0]

df_both = pd.DataFrame({
    'text':         df_r[text_col],
    'true_label':   df_r['true_label_int'],
    'pred_roberta': df_r['pred_label_int'],
    'pred_ollama':  df_z['pred_label_int'],
    'prob_rumour_roberta': y_prob_r,
    'conf_ollama':  y_conf_z,
})

df_both['roberta_correct'] = df_both['pred_roberta'] == df_both['true_label']
df_both['ollama_correct']  = df_both['pred_ollama']  == df_both['true_label']
df_both['agree']           = df_both['pred_roberta'] == df_both['pred_ollama']

both_correct = df_both[ df_both['roberta_correct'] &  df_both['ollama_correct']]
both_wrong   = df_both[~df_both['roberta_correct'] & ~df_both['ollama_correct']]
only_roberta = df_both[ df_both['roberta_correct'] & ~df_both['ollama_correct']]
only_ollama  = df_both[~df_both['roberta_correct'] &  df_both['ollama_correct']]

print(f"Both correct:        {len(both_correct):4d} ({len(both_correct)/len(df_both)*100:.1f}%)")
print(f"Only RoBERTa right:  {len(only_roberta):4d} ({len(only_roberta)/len(df_both)*100:.1f}%)")
print(f"Only Ollama right:   {len(only_ollama):4d} ({len(only_ollama)/len(df_both)*100:.1f}%)")
print(f"Both wrong:          {len(both_wrong):4d} ({len(both_wrong)/len(df_both)*100:.1f}%)")
print(f"\nAgreement rate: {df_both['agree'].mean()*100:.1f}%")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
cats   = ['Both Correct', 'Only RoBERTa', 'Only Ollama', 'Both Wrong']
counts = [len(both_correct), len(only_roberta), len(only_ollama), len(both_wrong)]
cols   = ['#27ae60', COLORS['roberta'], COLORS['ollama'], '#c0392b']
bars   = ax.bar(cats, counts, color=cols, edgecolor='black', linewidth=0.6, alpha=0.85)
for bar, cnt in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            str(cnt), ha='center', va='bottom', fontweight='bold')
ax.set_ylabel('Number of Tweets')
ax.set_title('PHEME — Prediction Agreement Analysis', fontweight='bold')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(FIGS_DIR / 'pheme_agreement.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Sample Tweets — Where Models Disagree

In [ ]:
id2label = {0: 'not_rumour', 1: 'rumour'}

print("--- Only RoBERTa Got Right (Ollama missed) ---")
for _, row in only_roberta.head(4).iterrows():
    print(f"  True: {id2label[row['true_label']]:12s} | RoBERTa: ✓ | Ollama: ✗")
    print(f"  Tweet: {str(row['text'])[:110]}")
    print()

print("\n--- Only Ollama Got Right (RoBERTa missed) ---")
for _, row in only_ollama.head(4).iterrows():
    print(f"  True: {id2label[row['true_label']]:12s} | RoBERTa: ✗ | Ollama: ✓")
    print(f"  Tweet: {str(row['text'])[:110]}")
    print()

## 9. Final Summary Table

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.axis('off')

summary_metrics = ['Accuracy', 'F1 Macro', 'F1 Weighted',
                   f'F1 ({POS_LABEL})', f'Precision ({POS_LABEL})', f'Recall ({POS_LABEL})']

table_data = []
for m in summary_metrics:
    r_val = metrics_r[m]
    z_val = metrics_z[m]
    delta = r_val - z_val
    winner = '🔵 RoBERTa' if delta > 0.001 else ('🟠 Ollama' if delta < -0.001 else 'Tie')
    table_data.append([m, f'{r_val:.4f}', f'{z_val:.4f}', f'{delta:+.4f}', winner])

col_labels = ['Metric', 'RoBERTa', 'Ollama', 'Delta (R-O)', 'Winner']
table = ax.table(cellText=table_data, colLabels=col_labels,
                 cellLoc='center', loc='center', bbox=[0, 0, 1, 1])
table.auto_set_font_size(False)
table.set_fontsize(10)
for (row, col), cell in table.get_celld().items():
    if row == 0:
        cell.set_facecolor('#2c3e50')
        cell.set_text_props(color='white', fontweight='bold')
    elif row % 2 == 0:
        cell.set_facecolor('#f2f3f4')

ax.set_title('PHEME — Final Comparison: RoBERTa vs Ollama Zero-Shot',
             fontsize=13, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig(FIGS_DIR / 'pheme_final_summary.png', dpi=150, bbox_inches='tight')
plt.show()